[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/automatizacion/03-pipelines-negocio.ipynb)

# Pipelines de Negocio con IA

> Construye pipelines de negocio completos que integran LLMs para automatizar
> procesos de ventas, RRHH, finanzas y atención al cliente.

## Objetivos
- Automatizar el pipeline de ventas: lead → propuesta → contrato
- Procesar solicitudes de RRHH con IA
- Analizar documentos financieros automáticamente
- Implementar escalado inteligente a agentes humanos

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic pandas --quiet

## 2. Pipeline de Ventas: Lead → Propuesta → Follow-up

In [ ]:
import anthropic
import json
import pandas as pd
from datetime import datetime, timedelta

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

# Catálogo de productos/servicios
CATALOGO = """
Productos disponibles:
- Plan Básico: 299€/mes — hasta 3 usuarios, 10K llamadas API, soporte email
- Plan Profesional: 799€/mes — hasta 20 usuarios, 100K llamadas API, soporte 24/7
- Plan Enterprise: desde 2500€/mes — usuarios ilimitados, SLA 99.9%, integración personalizada
- Implementación: 1500-5000€ según complejidad
- Formación: 800€/día (equipos de hasta 10 personas)
"""

def paso_calificar_lead(lead: dict) -> dict:
    """Califica el lead y determina el plan más adecuado."""
    prompt = f"""Eres un consultor de ventas experto. Califica este lead.

Lead:
- Empresa: {lead['empresa']} ({lead['empleados']} empleados)
- Sector: {lead['sector']}
- Necesidad: {lead['necesidad']}
- Presupuesto indicativo: {lead.get('presupuesto', 'no indicado')}

Catálogo: {CATALOGO}

Responde con JSON:
{{"score_lead": 1-10,
  "plan_recomendado": "Básico|Profesional|Enterprise",
  "razon": "<justificación en 1 frase>",
  "servicios_adicionales": ["implementación"|"formación"],
  "valor_estimado_anual": 0}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=250,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json")
    return {**lead, **json.loads(resp)}

def paso_generar_propuesta(lead_calificado: dict) -> dict:
    """Genera propuesta comercial personalizada."""
    prompt = f"""Genera una propuesta comercial personalizada en español.

Cliente: {lead_calificado['empresa']} ({lead_calificado['sector']})
Necesidad: {lead_calificado['necesidad']}
Plan: {lead_calificado['plan_recomendado']}
Servicios adicionales: {', '.join(lead_calificado.get('servicios_adicionales', []))}
Catálogo: {CATALOGO}

Genera una propuesta ejecutiva en markdown (máx 200 palabras) que incluya:
- Resumen ejecutivo (problema y solución)
- Plan recomendado con precio
- ROI estimado
- Próximos pasos"""

    propuesta = client.messages.create(
        model=MODELO, max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text
    return {**lead_calificado, "propuesta": propuesta}

def paso_generar_followup(lead_con_propuesta: dict, dias: int = 3) -> str:
    """Genera email de seguimiento post-propuesta."""
    fecha_followup = (datetime.now() + timedelta(days=dias)).strftime("%d/%m/%Y")
    prompt = f"""Genera un email de seguimiento breve y no invasivo para {lead_con_propuesta['empresa']}.
El email se enviará el {fecha_followup} (3 días después de la propuesta).
El contacto es {lead_con_propuesta.get('contacto', 'el responsable')}.
No suenes desesperado. Máximo 5 líneas."""
    return client.messages.create(
        model=MODELO, max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text

LEADS = [
    {"empresa": "Distribuidora Pérez SA", "empleados": 45, "sector": "Distribución",
     "contacto": "Javier Pérez", "necesidad": "Automatizar pedidos y gestión de almacén",
     "presupuesto": "hasta 1000€/mes"},
    {"empresa": "Hospital San Rafael", "empleados": 800, "sector": "Salud",
     "contacto": "Dra. López", "necesidad": "Digitalizar informes médicos y atención al paciente",
     "presupuesto": "amplio, sin límite definido"},
]

print("=== PIPELINE DE VENTAS ===")
for lead in LEADS:
    print(f"\nProcesando: {lead['empresa']}")
    calificado = paso_calificar_lead(lead)
    print(f"  Score: {calificado['score_lead']}/10 | Plan: {calificado['plan_recomendado']} | VAA: {calificado.get('valor_estimado_anual', 0):,}€")
    con_propuesta = paso_generar_propuesta(calificado)
    print(f"  Propuesta generada ({len(con_propuesta['propuesta'])} chars)")
    followup = paso_generar_followup(con_propuesta)
    print(f"  Follow-up: {followup[:100]}...")

## 3. Pipeline RRHH: Procesamiento de solicitudes

In [ ]:
SOLICITUDES_RRHH = [
    {"id": "HR001", "empleado": "María García", "tipo": "vacaciones",
     "texto": "Solicito 10 días de vacaciones del 15 al 26 de julio"},
    {"id": "HR002", "empleado": "Pedro Sánchez", "tipo": "formación",
     "texto": "Quiero asistir al curso de Python avanzado en Madrid el 5 y 6 de mayo, coste 600€"},
    {"id": "HR003", "empleado": "Ana Torres", "tipo": "teletrabajo",
     "texto": "Solicito ampliar el teletrabajo de 2 a 4 días por semana por motivos de conciliación familiar"},
]

POLITICAS_RRHH = """
Políticas:
- Vacaciones: máx 30 días/año, solicitar con 15 días de antelación
- Formación: hasta 1500€/año por empleado, aprobación del departamento
- Teletrabajo: máx 3 días/semana según convenio, aprobación de manager
"""

def procesar_solicitud_rrhh(solicitud: dict) -> dict:
    """Procesa solicitud RRHH: extrae datos, verifica política, recomienda decisión."""
    prompt = f"""Procesa esta solicitud de RRHH según las políticas de la empresa.

Solicitud de {solicitud['empleado']} (ID: {solicitud['id']}):
{solicitud['texto']}

Políticas: {POLITICAS_RRHH}

Responde con JSON:
{{"tipo_detectado": "<tipo>",
  "cumple_politica": true/false,
  "decision_recomendada": "aprobar|rechazar|revisar_manual",
  "motivo": "<explicación>",
  "respuesta_empleado": "<email de respuesta breve al empleado>"}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json")
    return {**solicitud, **json.loads(resp)}

print("=== PIPELINE RRHH ===")
iconos = {"aprobar": "✅", "rechazar": "❌", "revisar_manual": "⚠️"}
for s in SOLICITUDES_RRHH:
    resultado = procesar_solicitud_rrhh(s)
    icono = iconos.get(resultado['decision_recomendada'], "❓")
    print(f"\n{icono} {resultado['id']} — {resultado['empleado']}")
    print(f"   Tipo: {resultado['tipo_detectado']} | Política: {'✓' if resultado['cumple_politica'] else '✗'} | Decisión: {resultado['decision_recomendada']}")
    print(f"   Motivo: {resultado['motivo']}")

## 4. Pipeline Financiero: Análisis de facturas

In [ ]:
FACTURAS_TEXTO = [
    {"id": "F2024-001", "texto": "FACTURA de Oficinas García SL. Proveedor: Suministros Madrid SA (NIF B-12345678). Concepto: Material de oficina (papel A4 × 10 resmas, bolígrafos × 50, carpetas × 20). Total: 287,50€ IVA incluido (21%). Fecha: 15/03/2024. Vencimiento: 45 días."},
    {"id": "F2024-002", "texto": "Factura nº 2024/156 de TechServ Solutions. Cliente: Nuestra empresa. Servicio: Mantenimiento mensual servidores cloud. Base imponible: 1.200€. IVA 21%: 252€. Total: 1.452€. Fecha servicio: marzo 2024. Pago: transferencia bancaria IBAN ES12 0000 1111 2222 3333."},
]

def analizar_factura(factura: dict) -> dict:
    """Extrae y valida datos de factura en texto libre."""
    prompt = f"""Extrae los datos estructurados de esta factura.

Factura: {factura['texto']}

Responde SOLO con JSON:
{{"proveedor": "<nombre>",
  "nif_proveedor": "<NIF o vacío>",
  "concepto": "<descripción>",
  "base_imponible": 0.0,
  "iva_porcentaje": 0,
  "total": 0.0,
  "fecha": "<fecha o vacío>",
  "categoria_contable": "suministros|servicios|tecnología|personal|otro",
  "alertas": ["<alerta si algo parece incorrecto>"]}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json")
    datos = json.loads(resp)
    # Validar coherencia IVA
    if datos["base_imponible"] > 0:
        iva_calculado = round(datos["base_imponible"] * datos["iva_porcentaje"] / 100, 2)
        total_calculado = round(datos["base_imponible"] + iva_calculado, 2)
        if abs(total_calculado - datos["total"]) > 0.02:
            datos["alertas"].append(f"Total no cuadra: {datos['base_imponible']} + IVA = {total_calculado}, declarado {datos['total']}")
    return {"id": factura["id"], **datos}

print("=== PIPELINE FINANCIERO ===")
facturas_procesadas = [analizar_factura(f) for f in FACTURAS_TEXTO]
df_facturas = pd.DataFrame(facturas_procesadas)
print(df_facturas[["id", "proveedor", "categoria_contable", "total", "alertas"]].to_string(index=False))
print(f"\nTotal facturas: {df_facturas['total'].sum():.2f}€")

## 5. Sistema de escalado inteligente

In [ ]:
def sistema_escalado(solicitud: str, contexto: str = "") -> dict:
    """Decide si una solicitud puede resolverse automáticamente o requiere humano."""
    prompt = f"""Eres un sistema de triaje para soporte empresarial.

Solicitud: {solicitud}
Contexto adicional: {contexto or 'ninguno'}

Determina:
1. ¿Puede resolverse automáticamente?
2. Si sí, genera la resolución
3. Si no, identifica el especialista necesario

Responde con JSON:
{{"puede_auto_resolver": true/false,
  "confianza": 0.0-1.0,
  "resolucion_auto": "<respuesta si puede resolverse>",
  "motivo_escalado": "<si necesita humano>",
  "especialista": "facturación|legal|técnico|dirección|ninguno",
  "urgencia": "inmediata|24h|72h|esta_semana"}}"""

    resp = client.messages.create(
        model=MODELO, max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if "```" in resp:
        resp = resp.split("```")[1].lstrip("json")
    return json.loads(resp)

CASOS = [
    ("¿Cuál es el horario de atención al cliente?", ""),
    ("Necesito una factura rectificativa por error en el IVA del año pasado", "importe: 15.000€"),
    ("El sistema no carga, página en blanco", "usuario: premium, navegador: Chrome"),
    ("Amenaza con demanda por supuesta filtración de datos personales", "cliente desde 2019"),
]

print("=== SISTEMA DE ESCALADO INTELIGENTE ===")
for solicitud, contexto in CASOS:
    resultado = sistema_escalado(solicitud, contexto)
    modo = "🤖 AUTO" if resultado["puede_auto_resolver"] else f"👤 ESCALADO → {resultado['especialista'].upper()}"
    print(f"\n{modo} [{resultado['urgencia']}] (confianza: {resultado['confianza']:.0%})")
    print(f"   Solicitud: {solicitud[:60]}")
    if resultado["puede_auto_resolver"]:
        print(f"   Respuesta: {resultado['resolucion_auto'][:80]}...")
    else:
        print(f"   Motivo escalado: {resultado['motivo_escalado']}")